In [1]:
import os
import pandas as pd
import numpy as np


def extract_sample_id_from_annotations(annotations: pd.DataFrame,
                                      file_name: str) -> str:
    """
    Given the annotations table and a CNV filename, extract the 'real'
    sample ID using the same logic as your SV code.

    Expects annotations columns:
        'File Name', 'Tumor Descriptor', 'Sample ID'
    """
    row = annotations[annotations["File Name"] == file_name]
    if row.empty:
        raise ValueError(f"No annotation row found for file_name={file_name!r}")

    row = row.iloc[0]
    tumor_desc = str(row["Tumor Descriptor"])
    sample_id_field = str(row["Sample ID"])
    parts = [p.strip() for p in sample_id_field.split(",")]

    if tumor_desc.startswith("Not"):
        # use the second ID if tumor descriptor says "Not ..."
        if len(parts) < 2:
            raise ValueError(f"Expected 2 sample IDs in {sample_id_field!r}")
        sample_id = parts[1]
    else:
        sample_id = parts[0]

    return sample_id


def compute_signed_distance_with_overlap(del_start, del_end, gene_start, gene_end, strand):
    """
    Compute signed distance between deletion [del_start, del_end]
    and gene [gene_start, gene_end], and also return overlap positions.

    Returns:
        signed_distance, overlap_start, overlap_end
        - signed_distance = 0 if overlapping
        - overlap_start/end = genomic coordinates of overlap, else None
    """

    # --- Check overlap ---
    if del_end >= gene_start and del_start <= gene_end:
        overlap_start = max(del_start, gene_start)
        overlap_end = min(del_end, gene_end)

        return 0, overlap_start, overlap_end

    # --- No overlap ---
    overlap_start = None
    overlap_end = None

    # deletion completely before gene
    if del_end < gene_start:
        d = gene_start - del_end
        if strand == '+':
            return -d, overlap_start, overlap_end
        else:
            return +d, overlap_start, overlap_end

    # deletion completely after gene
    if del_start > gene_end:
        d = del_start - gene_end
        if strand == '+':
            return +d, overlap_start, overlap_end
        else:
            return -d, overlap_start, overlap_end

    # Should not reach here
    return 0, overlap_start, overlap_end



def add_basic_cnv_features(cnv_df: pd.DataFrame) -> pd.DataFrame:
    """
    Take a raw CNV table with columns:
      GDC_Aliquot, Chromosome, Start, End,
      Copy_Number, Major_Copy_Number, Minor_Copy_Number
    and add derived features.
    """
    df = cnv_df.copy()

    df["cn_total"] = df["Copy_Number"].astype(float)
    df["cn_major"] = df["Major_Copy_Number"].astype(float)
    df["cn_minor"] = df["Minor_Copy_Number"].astype(float)

    df["loh_flag"] = (df["cn_minor"] == 0).astype(int)

    def classify_state(cn):
        if pd.isna(cn):
            return "NA"
        cn = float(cn)
        if cn == 0:
            return "deep_del"
        elif cn == 1:
            return "loss"
        elif cn == 2:
            return "neutral"
        elif cn == 3:
            return "gain"
        elif cn >= 4:
            return "amp"
        else:
            return "other"

    df["cn_state"] = df["cn_total"].map(classify_state)
    df["segment_len"] = df["End"] - df["Start"] + 1

    return df


def classify_span_sv_gene_hit(del_start: int,
                          del_end: int,
                          gene_start: int,
                          gene_end: int,
                          strand: str,
                          signed_dist: int,
                          gene_features: pd.DataFrame,
                          promoter_up: int = 2000,
                          promoter_down: int = 500,
                          proximal_window: int = 5000) -> dict:
    """
    Return a dict with region-level annotations for a DEL–gene pair.
    gene_features: rows from `genes` for this gene (exon/CDS/UTR etc.).
    """

    # --- gene body overlap ---
    inter_body_start = max(del_start, gene_start)
    inter_body_end = min(del_end, gene_end)
    if inter_body_end >= inter_body_start:
        overlap_bp = inter_body_end - inter_body_start + 1
        overlap_percent = (overlap_bp / (gene_end - gene_start + 1)) * 100
        gene_body_flag = 1
    else:
        overlap_bp = 0
        overlap_percent = 0
        gene_body_flag = 0

    # --- promoter region (strand-aware) ---
    # TSS = gene_start on + strand, gene_end on - strand
    if strand == "+":
        tss = gene_start
        prom_start = tss - promoter_up
        prom_end = tss + promoter_down
    else:
        tss = gene_end
        prom_start = tss - promoter_down
        prom_end = tss + promoter_up

    inter_prom_start = max(del_start, prom_start)
    inter_prom_end = min(del_end, prom_end)
    promoter_flag = 1 if inter_prom_end >= inter_prom_start else 0

    # --- detailed gene-structure hits from `genes` ---
    hits_exon_like = False
    hits_utr = False
    hits_stop_codon = False
    hits_start_codon = False
    transcript_id = None
    transcript_type = None
    if gene_features is not None and not gene_features.empty:
        for _, feat in gene_features.iterrows():
            f_start = int(feat["start"])
            f_end = int(feat["end"])
            if del_end < f_start or del_start > f_end:
                continue  # no overlap
            ftype = feat["feature"]
            if ftype in ("exon", "CDS", "start_codon", "stop_codon", "transcript"):
                hits_exon_like = True
                if ftype == "stop_codon":
                    hits_stop_codon = True
                if ftype == "start_codon":
                    hits_start_codon = True
                elif ftype == "transcript":
                    transcript_id = feat["transcript_id"]
                    transcript_type = feat["transcript_type"]
            elif ftype == "UTR":
                hits_utr = True


    exon_flag = 1 if (hits_exon_like or hits_utr) else 0
    intron_only_flag = 1 if (gene_body_flag and not exon_flag) else 0
    stop_codon_flag = 1 if hits_stop_codon else 0
    start_codon_flag = 1 if hits_start_codon else 0

    # --- classify region_hit (coarse) ---
    region_hit = ""
    if promoter_flag or overlap_bp > 0:
        if promoter_flag:
            region_hit += "promoter,"
        if overlap_bp > 0:
            if exon_flag:
                region_hit += "exon_or_UTR,"
            else:
                region_hit += "intron,"
    else:
        # no gene body overlap → maybe proximal upstream/downstream
        if abs(signed_dist) <= proximal_window:
            if signed_dist < 0:
                region_hit += "upstream_5kb,"
            else:
                region_hit += "downstream_5kb,"
        else:
            region_hit += "intergenic"

    upstream_5kb_flag = int("upstream_5kb" in region_hit)
    downstream_5kb_flag = int("downstream_5kb" in region_hit)


    return {
        "overlap_bp": overlap_bp,
        "overlap_percent": overlap_percent,
        "promoter_flag": promoter_flag,
        "gene_body_flag": gene_body_flag,
        "exon_flag": exon_flag,
        "intron_only_flag": intron_only_flag,
        "upstream_5kb_flag": upstream_5kb_flag,
        "downstream_5kb_flag": downstream_5kb_flag,
        "region_hit": region_hit,
        "stop_codon_flag": stop_codon_flag,
        "start_codon_flag": start_codon_flag,
        "transcript_id" : transcript_id,
        "transcript_type": transcript_type
    }





def annotate_cnv_with_gene_hits(
    cnv_df: pd.DataFrame,
    genes_all_features_df: pd.DataFrame,
    promoter_up: int = 2000,
    promoter_down: int = 500,
    proximal_window: int = 5000,
) -> pd.DataFrame:
    """
    Add a `gene_hits` column (list[dict]) to the CNV segments, using the
    same helpers as span SVs.

    genes_all_features_df: the full "primary_genes_all_features.csv"
    (after any frame filtering), with rows for gene, exon, CDS, UTR, etc.
    """

    df = cnv_df.copy()
    df["Chromosome"] = df["Chromosome"].astype(str)

    genes_all = genes_all_features_df.copy()
    genes_all["chrom"] = genes_all["chrom"].astype(str)

    # gene-level rows
    genes_gene = genes_all[genes_all["feature"] == "gene"].copy()

    # Pre-allocate list-of-lists
    gene_hits_col = [[] for _ in range(len(df))]

    # Group CNV segments by chromosome
    idx_by_chrom = df.groupby("Chromosome").indices

    for chrom, idxs in idx_by_chrom.items():
        seg_idxs = list(idxs)
        segs = df.loc[seg_idxs]

        genes_c = genes_gene[genes_gene["chrom"] == chrom]
        if genes_c.empty:
            continue

        gene_feats_c = genes_all[genes_all["chrom"] == chrom]

        genes_c = genes_c.sort_values(["start", "end"])

        for seg_idx, seg_row in segs.iterrows():
            seg_start = int(seg_row["Start"])
            seg_end   = int(seg_row["End"])

            # coarse prefilter in coordinate space (± proximal_window)
            candidates = genes_c[
                (genes_c["end"]   >= seg_start - proximal_window) &
                (genes_c["start"] <= seg_end   + proximal_window)
            ]
            if candidates.empty:
                continue

            seg_hits = []

            for _, g in candidates.iterrows():
                gene_start = int(g["start"])
                gene_end   = int(g["end"])
                strand     = g["strand"]

                signed_dist, _, _ = compute_signed_distance_with_overlap(
                    del_start=seg_start,
                    del_end=seg_end,
                    gene_start=gene_start,
                    gene_end=gene_end,
                    strand=strand,
                )

                # all features for this gene (exon/CDS/UTR/tx, etc.)
                g_feats = gene_feats_c[gene_feats_c["gene_name"] == g["gene_name"]]

                ann = classify_span_sv_gene_hit(
                    del_start=seg_start,
                    del_end=seg_end,
                    gene_start=gene_start,
                    gene_end=gene_end,
                    strand=strand,
                    signed_dist=signed_dist,
                    gene_features=g_feats,
                    promoter_up=promoter_up,
                    promoter_down=promoter_down,
                    proximal_window=proximal_window,
                )

                # skip far intergenic genes
                if ann["region_hit"] == "intergenic":
                    continue

                hit = {
                    "gene_name": g["gene_name"],
                    "gene_id":   g.get("gene_id"),
                    "strand":    strand,
                    "signed_dist": signed_dist,
                    **ann,
                }
                seg_hits.append(hit)

            gene_hits_col[seg_idx] = seg_hits

    df["gene_hits"] = gene_hits_col
    return df


def annotate_cnv_with_mirna_hits(
    cnv_df: pd.DataFrame,
    mirna_df: pd.DataFrame,
    promoter_up: int = 2000,
    promoter_down: int = 500,
    proximal_window: int = 5000,
) -> pd.DataFrame:
    """
    Add a `gene_hits` column (list[dict]) to the CNV segments, using the
    same helpers as span SVs.

    genes_all_features_df: the full "primary_genes_all_features.csv"
    (after any frame filtering), with rows for gene, exon, CDS, UTR, etc.
    """

    df = cnv_df.copy()
    df["Chromosome"] = df["Chromosome"].astype(str)

    mirna_df["chrom"] = mirna_df["chrom"].astype(str)


    # Pre-allocate list-of-lists
    mir_hits_col = [[] for _ in range(len(df))]

    # Group CNV segments by chromosome
    idx_by_chrom = df.groupby("Chromosome").indices

    for chrom, idxs in idx_by_chrom.items():
        seg_idxs = list(idxs)
        segs = df.loc[seg_idxs]

        mirna_c = mirna_df[mirna_df["chrom"] == chrom]
        if mirna_c.empty:
            continue


        mirna_c = mirna_c.sort_values(["start", "end"])

        for seg_idx, seg_row in segs.iterrows():
            seg_start = int(seg_row["Start"])
            seg_end   = int(seg_row["End"])

            # coarse prefilter in coordinate space (± proximal_window)
            candidates = mirna_c[
                (mirna_c["end"]   >= seg_start - proximal_window) &
                (mirna_c["start"] <= seg_end   + proximal_window)
            ]
            if candidates.empty:
                continue

            seg_hits = []

            for _, g in candidates.iterrows():
                gene_start = int(g["start"])
                gene_end   = int(g["end"])
                strand     = g["strand"]

                signed_dist, _, _ = compute_signed_distance_with_overlap(
                    del_start=seg_start,
                    del_end=seg_end,
                    gene_start=gene_start,
                    gene_end=gene_end,
                    strand=strand,
                )

                # all features for this gene (exon/CDS/UTR/tx, etc.)

                ann = classify_span_sv_gene_hit(
                    del_start=seg_start,
                    del_end=seg_end,
                    gene_start=gene_start,
                    gene_end=gene_end,
                    strand=strand,
                    signed_dist=signed_dist,
                    gene_features=None,
                    promoter_up=promoter_up,
                    promoter_down=promoter_down,
                    proximal_window=proximal_window,
                )

                # skip far intergenic genes
                if ann["region_hit"] == "intergenic":
                    continue

                hit = {
                    "gene_name": g["gene_name"],
                    "gene_id":   g.get("gene_id"),
                    "strand":    strand,
                    "signed_dist": signed_dist,
                    **ann,
                }
                seg_hits.append(hit)

            mir_hits_col[seg_idx] = seg_hits

    df["mir_hits"] = mir_hits_col
    return df


def annotate_cnv_with_lncRNAs_hits(
    cnv_df: pd.DataFrame,
    lncRNAs_all_features_df: pd.DataFrame,
    promoter_up: int = 2000,
    promoter_down: int = 500,
    proximal_window: int = 5000,
) -> pd.DataFrame:
    """
    Add a `gene_hits` column (list[dict]) to the CNV segments, using the
    same helpers as span SVs.

    genes_all_features_df: the full "primary_genes_all_features.csv"
    (after any frame filtering), with rows for gene, exon, CDS, UTR, etc.
    """

    df = cnv_df.copy()
    df["Chromosome"] = df["Chromosome"].astype(str)

    lncRNAs_all = lncRNAs_all_features_df.copy()
    lncRNAs_all["chrom"] = lncRNAs_all["chrom"].astype(str)

    # gene-level rows
    lncRNAs_gene = lncRNAs_all[lncRNAs_all["feature"] == "gene"].copy()

    # Pre-allocate list-of-lists
    lncRNAs_hits_col = [[] for _ in range(len(df))]

    # Group CNV segments by chromosome
    idx_by_chrom = df.groupby("Chromosome").indices

    for chrom, idxs in idx_by_chrom.items():
        seg_idxs = list(idxs)
        segs = df.loc[seg_idxs]

        lncRNAs_c = lncRNAs_gene[lncRNAs_gene["chrom"] == chrom]
        if lncRNAs_c.empty:
            continue

        lncRNAs_feats_c = lncRNAs_all[lncRNAs_all["chrom"] == chrom]

        lncRNAs_c = lncRNAs_c.sort_values(["start", "end"])

        for seg_idx, seg_row in segs.iterrows():
            seg_start = int(seg_row["Start"])
            seg_end   = int(seg_row["End"])

            # coarse prefilter in coordinate space (± proximal_window)
            candidates = lncRNAs_c[
                (lncRNAs_c["end"]   >= seg_start - proximal_window) &
                (lncRNAs_c["start"] <= seg_end   + proximal_window)
            ]
            if candidates.empty:
                continue

            seg_hits = []

            for _, g in candidates.iterrows():
                gene_start = int(g["start"])
                gene_end   = int(g["end"])
                strand     = g["strand"]

                signed_dist, _, _ = compute_signed_distance_with_overlap(
                    del_start=seg_start,
                    del_end=seg_end,
                    gene_start=gene_start,
                    gene_end=gene_end,
                    strand=strand,
                )

                # all features for this gene (exon/CDS/UTR/tx, etc.)
                g_feats = lncRNAs_feats_c[lncRNAs_feats_c["gene_name"] == g["gene_name"]]

                ann = classify_span_sv_gene_hit(
                    del_start=seg_start,
                    del_end=seg_end,
                    gene_start=gene_start,
                    gene_end=gene_end,
                    strand=strand,
                    signed_dist=signed_dist,
                    gene_features=g_feats,
                    promoter_up=promoter_up,
                    promoter_down=promoter_down,
                    proximal_window=proximal_window,
                )

                # skip far intergenic genes
                if ann["region_hit"] == "intergenic":
                    continue

                hit = {
                    "gene_name": g["gene_name"],
                    "gene_id":   g.get("gene_id"),
                    "strand":    strand,
                    "signed_dist": signed_dist,
                    **ann,
                }
                seg_hits.append(hit)

            lncRNAs_hits_col[seg_idx] = seg_hits

    df["lncRNAs_hits"] = lncRNAs_hits_col
    return df

def compute_signed_distance_interval(
    seg_start: int,
    seg_end: int,
    elem_start: int,
    elem_end: int,
) -> int:
    """
    Signed distance between segment [seg_start, seg_end]
    and element [elem_start, elem_end] on coordinate axis.
    """
    # overlap
    if seg_end >= elem_start and seg_start <= elem_end:
        return 0

    # seg completely before element
    if seg_end < elem_start:
        return -(elem_start - seg_end)

    # seg completely after element
    if seg_start > elem_end:
        return seg_start - elem_end

    return 0


def classify_span_sv_elem_hit(sv_start: int,
                              sv_end: int,
                              elem_start: int,
                              elem_end: int,
                              signed_dist: int,
                              proximal_window: int = 5000) -> dict:
    """
    Return a dict with region-level annotations for a span-SV–element pair.

    This is simpler than the gene version: no strand, no promoter/exon logic.
    """

    # --- element overlap ---
    inter_start = max(sv_start, elem_start)
    inter_end = min(sv_end, elem_end)
    if inter_end >= inter_start:
        overlap_bp = inter_end - inter_start + 1
        overlap_percent = (overlap_bp / (elem_end - elem_start + 1)) * 100
        overlaps_flag = 1
    else:
        overlap_bp = 0
        overlaps_flag = 0
        overlap_percent = 0

    # --- classify coarse region_hit ---
    region_hit = ""
    if overlaps_flag:
        region_hit = "overlaps"
    else:
        if abs(signed_dist) <= proximal_window:
            # You can keep "upstream/downstream" semantics from signed_dist
            if signed_dist < 0:
                region_hit = "proximal_upstream"
            elif signed_dist > 0:
                region_hit = "proximal_downstream"
            else:
                region_hit = "proximal"  # degenerate case
        else:
            region_hit = "distal"

    proximal_flag = int("proximal" in region_hit)
    distal_flag = int("distal" in region_hit)


    return {
        "overlap_bp": overlap_bp,
        "overlap_percent": overlap_percent,
        "overlaps_flag": overlaps_flag,
        "region_hit": region_hit,
        "proximal_flag": proximal_flag,
        "distal_flag": distal_flag,
    }


def annotate_cnv_with_elem_hits(
    cnv_df: pd.DataFrame,
    elements_df: pd.DataFrame,
    proximal_window: int = 5000,
) -> pd.DataFrame:
    """
    Add `elem_hits` column (list[dict]) to CNV segments using
    classify_span_sv_elem_hit.

    elements_df must have:
        element_id, chrom, start, end
    (and optionally element_type, etc.)
    """

    df = cnv_df.copy()
    df["Chromosome"] = df["Chromosome"].astype(str)

    elems = elements_df.copy()
    elems["chrom"] = elems["chrom"].astype(str)

    elem_hits_col = [[] for _ in range(len(df))]

    idx_by_chrom = df.groupby("Chromosome").indices

    for chrom, idxs in idx_by_chrom.items():
        seg_idxs = list(idxs)
        segs = df.loc[seg_idxs]

        elems_c = elems[elems["chrom"] == chrom]
        if elems_c.empty:
            continue

        elems_c = elems_c.sort_values(["start", "end"])

        for seg_idx, seg_row in segs.iterrows():
            seg_start = int(seg_row["Start"])
            seg_end   = int(seg_row["End"])

            candidates = elems_c[
                (elems_c["end"]   >= seg_start - proximal_window) &
                (elems_c["start"] <= seg_end   + proximal_window)
            ]
            if candidates.empty:
                continue

            seg_hits = []

            for _, e in candidates.iterrows():
                elem_start = int(e["start"])
                elem_end   = int(e["end"])

                signed_dist = compute_signed_distance_interval(
                    seg_start, seg_end,
                    elem_start, elem_end,
                )

                ann = classify_span_sv_elem_hit(
                    sv_start=seg_start,
                    sv_end=seg_end,
                    elem_start=elem_start,
                    elem_end=elem_end,
                    signed_dist=signed_dist,
                    proximal_window=proximal_window,
                )

                # keep overlapping / proximal, drop distal
                if ann["region_hit"] == "distal":
                    continue

                hit = {
                    "element_id":   e["elem_id"],
                    "element_type": e.get("element_type"),
                    "signed_dist":  signed_dist,
                    **ann,
                }
                seg_hits.append(hit)

            elem_hits_col[seg_idx] = seg_hits

    df["elem_hits"] = elem_hits_col
    return df


def process_cnv_directory(
    cnv_dir: str,
    genes_path: str,
    lncRNAs_path: str,
    mirna_path: str,
    elements_path: str,
    sample_annotations_path: str,
    output_dir: str,
    cnv_sep: str = "\t",
    promoter_up: int = 2000,
    promoter_down: int = 500,
    proximal_window: int = 5000,
    file_suffix_filter: tuple = (".txt", ".tsv", ".seg"),
):
    """
    Full CNV processing wrapper.

    Steps per file:
      1. Read raw CNV table
      2. Add basic CNV features
      3. Annotate gene_hits and elem_hits
      4. Extract 'real' sample_id from annotations
      5. Add sample_id column and save to per-sample CSV
    """

    annotations = pd.read_csv(sample_annotations_path, sep="\t")

    # genes_all_features: same file you used for SVs
    genes_all = pd.read_csv(genes_path)
    genes_all = genes_all[genes_all["frame"].isin(["0", "."])]

    # lncRNAs_all: same file you used for SVs
    lncRNAs_all = pd.read_csv(lncRNAs_path)
    lncRNAs_all = lncRNAs_all[lncRNAs_all["frame"].isin(["0", "."])]

    # miRNA
    mirna_df = pd.read_csv(mirna_path)

    # CCREs
    elements = pd.read_csv(elements_path)

    os.makedirs(output_dir, exist_ok=True)

    for fname in os.listdir(cnv_dir):
        # optionally filter by suffix
        if file_suffix_filter is not None and not fname.endswith(file_suffix_filter):
            continue

        cnv_path = os.path.join(cnv_dir, fname)
        print(f"Processing CNV file: {fname}")

        # 1. read raw CNV
        cnv_raw = pd.read_csv(cnv_path, sep=cnv_sep)

        # 2. add CNV features
        cnv = add_basic_cnv_features(cnv_raw)

        # 3. annotate genes & elements
        cnv = annotate_cnv_with_gene_hits(
            cnv,
            genes_all_features_df=genes_all,
            promoter_up=promoter_up,
            promoter_down=promoter_down,
            proximal_window=proximal_window,
        )

        cnv = annotate_cnv_with_lncRNAs_hits(
            cnv,
            lncRNAs_all_features_df=lncRNAs_all,
            promoter_up=promoter_up,
            promoter_down=promoter_down,
            proximal_window=proximal_window,
        )

        cnv = annotate_cnv_with_mirna_hits(
            cnv,
            mirna_df=mirna_df,
            promoter_up=promoter_up,
            promoter_down=promoter_down,
            proximal_window=proximal_window,
        )

        cnv = annotate_cnv_with_elem_hits(
            cnv,
            elements_df=elements,
            proximal_window=proximal_window,
        )

        # 4. extract real sample ID
        # Here we assume the 'File Name' column in samples.tsv
        # contains the CNV file name exactly (i.e. same as `fname`).
        sample_id = extract_sample_id_from_annotations(annotations, file_name=fname)
        print(f"  → sample_id: {sample_id}")
        cnv.drop(columns=["GDC_Aliquot"], inplace=True)

        out_path = os.path.join(output_dir, f"{sample_id}_cnv_segments_annotated.csv")
        cnv.to_csv(out_path, index=False)

from pipeline.config import PATHS

CNV_DIR = PATHS.cnv_dir
GENES_PATH = PATHS.cnv_genes
lncRNAs_PATH = PATHS.lncrna_csv
MIRNA_PATH = PATHS.mirna_path
ELEMENTS_PATH = PATHS.regulatory_elements_table   # or similar
ANNOTATIONS_PATH = PATHS.cnv_annotations_path
OUTPUT_DIR = PATHS.cnv_output_dir

process_cnv_directory(
    cnv_dir=CNV_DIR,
    genes_path=GENES_PATH,
    lncRNAs_path=lncRNAs_PATH,
    mirna_path=MIRNA_PATH,
    elements_path=ELEMENTS_PATH,
    sample_annotations_path=ANNOTATIONS_PATH,
    output_dir=OUTPUT_DIR,
)





Processing CNV file: TCGA-BRCA.4695c0f4-4d30-4604-84b3-81407bf980e8.ascat3.allelic_specific.seg.txt
  → sample_id: TCGA-E9-A244-01A


In [2]:
import pandas as pd
s_proc = pd.read_csv(r"C:\Users\stavz\Desktop\masters\APM\data\CNV_TCGA\CNV_annotated\TCGA-E9-A244-01A_cnv_segments_annotated.csv")

In [4]:
s_proc["gene_hits"].iloc[4]

"[{'gene_name': 'JAK1', 'gene_id': 'ENSG00000162434.15', 'strand': '-', 'signed_dist': 0, 'overlap_bp': 234532, 'overlap_percent': 100.0, 'promoter_flag': 1, 'gene_body_flag': 1, 'exon_flag': 1, 'intron_only_flag': 0, 'upstream_5kb_flag': 0, 'downstream_5kb_flag': 0, 'region_hit': 'promoter,exon_or_UTR,', 'stop_codon_flag': 1, 'start_codon_flag': 1, 'transcript_id': 'ENST00000672099.1', 'transcript_type': 'protein_coding'}]"

In [4]:
elems = pd.read_csv(ELEMENTS_PATH)

In [8]:
s = pd.read_csv(f'{OUTPUT_DIR}\TCGA-E9-A244-01A_cnv_segments_annotated.csv')

In [3]:
from pipeline.config import PATHS
PATHS.cnv_dir

WindowsPath('/home/stavz/masters/gdc/APM/data/CNV_TCGA/CNV_extracted')